# Going Modular

**concept**: turn useful notebook code cells into reuseable python files.
we will need two notebooks:
1. cell mode: run as a traditional jupyter notebook/ google colab.
2. script mode: same as cell mode but with added functionality to turn useful code into python scripts, ex: train.py data_setup.py model_builder.py

cell mode: regular notebook. cells contain text or code.

difference b/w cell and script mode? cell mode is a more cleaned version of regular notebook of most useful code running one cell at a time.
script mode has extra subsection (e.g. 2.1 3.1...) for turning cell code into scipt code.

## 1. Get data

In [ ]:
!rm -rf models/

In [ ]:
import os
import zipfile

from pathlib import Path
import requests

# Setup path to data folder
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# If image folder doesn't exist, download it and prepare it
if image_path.is_dir():
  print(f"{image_path} directory exists.")
else:
  print(f"Did not find {image_path} directory, creating one...")
  image_path.mkdir(parents=True, exist_ok=True)

  # Download data
  with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/refs/heads/main/data/pizza_steak_sushi.zip")
    print("Downloading data...")
    f.write(request.content)

  # Unzip data
  with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
    print("Unzipping data....")
    zip_ref.extractall(image_path)

  # Remove zip file
  os.remove(data_path / "pizza_steak_sushi.zip")

In [ ]:
# setup train and testing paths
train_dir = image_path / "train"
test_dir = image_path / "test"

train_dir, test_dir

## 2. Create Datasets and Dataloaders

In [ ]:
from torchvision import datasets, transforms

# Create simple transforms
data_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

# Use image folder to create datasets
train_data = datasets.ImageFolder(root=train_dir,
                                  transform=data_transform,
                                  target_transform=None)

test_data = datasets.ImageFolder(root=test_dir,
                                  transform=data_transform)

print(f"Train data: \n{train_data}\nTest data: \n{test_data}")

In [ ]:
#Turn Train and Test Datasets into Dataloaders
from torch.utils.data import DataLoader
train_dataloader = DataLoader(dataset=train_data,
                              batch_size=1,
                              num_workers=1,
                              shuffle=True)

test_dataloader =  DataLoader(dataset=test_data,
                              batch_size=1,
                              num_workers=1,
                              shuffle=True)

train_dataloader, test_dataloader

In [ ]:
# Check out a single image size/shape
img, label = next(iter(train_dataloader))
print(f"Image shape: {img.shape} -> [batch, color, h, w]")
print(f"label shape: {label.shape}")

## 3. Making a model (TinyVGG)

In [ ]:
import torch
from torch import nn

class TinyVGG(nn.Module):
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
    super().__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels= input_shape,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)
    )
    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
        nn.ReLU(),
        nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_units*13*13,
                  out_features=output_shape)
    )

  def forward(self, x: torch.Tensor):
    x = self.conv_block_1(x)
    x = self.conv_block_2(x)
    x = self.classifier(x)
    return x

In [ ]:
import torch
device = "cude" if torch.cuda.is_available() else "cpu"

# Instantiate an instance of the model
torch.manual_seed(42)
model_0 = TinyVGG(input_shape=3,
                  hidden_units=10,
                  output_shape=len(train_data.classes)).to(device)

model_0

In [ ]:
# To test the model we do a single forward pass
img_batch, label_batch = next(iter(train_dataloader))

# 1. Get a batch of images and labels from the dataloader
img_0, label_0 = img_batch[0].unsqueeze(dim=0), label_batch[0]
print(f"Single image shape: {img_0.shape}\n")

# 2. Perform a forward pass on a single image
model_0.eval()

with torch.inference_mode():
  pred = model_0(img_0.to(device))

# 4. Print out what is happening and convert model logits -> pred probs -> labels
print(f"the pred logit: {pred}\n")
print(f"the pred prob: {torch.softmax(pred, dim=1)}\n")
print(f"the pred label: {torch.argmax(torch.softmax(pred, dim=1), dim=1)} \n")
print(f"Actual labe: {label_0}")

##4. Creating `train_step()` and `test_step` functions and combine them with `train()` function

`train_step()`

In [ ]:
from typing import Tuple

def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device) -> Tuple[float, float]: # it returns a tuple of loss and accuracy
  # Put model in train mode
  model.train()

  # Setup model loss and acc values
  train_loss, train_acc = 0, 0

  #loop through dataloader batchs
  for batch, (X, y) in enumerate(dataloader):
    # Send data to target device
    X, y = X.to(device), y.to(device)

    # 1. Forward pass
    y_pred = model(X)

    # 2. Calculate and accumulate the loss
    loss = loss_fn(y_pred, y)
    train_loss += loss.item()

    # 3. Optimizer zero grad
    optimizer.zero_grad()

    # 4. Loss backward
    loss.backward()

    # 5. Optimizer step
    optimizer.step()

    # Calculate and accumulate accuracy metric across all branches
    y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
    train_acc += (y_pred_class == y).sum().item() /len(y_pred)

  # Adjuct metrics to get average loss and accuracy per batch
  train_loss = train_loss / len(dataloader)
  train_acc = train_acc / len(dataloader)

  return train_loss, train_acc


`test_step()`

In [ ]:
def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device) -> Tuple[float, float]:
  # Put model in eval mode
  model.eval()

  # Setup test loss and accuracy values
  test_loss, test_acc = 0 , 0

  # Turn on inference context manager
  with torch.inference_mode():
    # Loop through DataLoader batches
    for batch, (X, y) in enumerate(dataloader):
      # Send data to target device
      X , y = X.to(device) , y.to(device)

      # 1. Forward pass
      test_pred_logit = model(X)

      # 2. Calculate and accumulate the loss
      loss = loss_fn(test_pred_logit, y)
      test_loss += loss.item()

      # Calculate and accumulate accuracy
      test_pred_label = test_pred_logit.argmax(dim=1)
      test_acc += ((test_pred_label == y).sum().item() / len(test_pred_label))

    # Adjuct metrics to get average loss and accuracy per batch
    test_loss += test_loss / len(dataloader)
    test_acc += test_acc / len(dataloader)

    return test_loss, test_acc

`train()`

In [ ]:
from typing import Dict, List
from tqdm.auto import tqdm

def train(model: torch.nn.Module,
               train_dataloader: torch.utils.data.DataLoader,
               test_dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               epochs: int,
               device: torch.device) -> Dict[str, List[float]]:

  # Empty result dictionary
  results = {"train_loss": [],
             "train_acc": [],
             "test_loss": [],
             "test_acc": []
  }

  # Loop through training and testing steps for a number of epochs
  for epoch in tqdm(range(epochs)):
    train_loss, train_acc = train_step(model = model,
                                      dataloader = train_dataloader,
                                      loss_fn = loss_fn,
                                      optimizer = optimizer,
                                      device = device)

    test_loss, test_acc = test_step(model = model,
                                      dataloader = test_dataloader,
                                      loss_fn = loss_fn,
                                      device = device)

    # Print out what is happening
    print(
        f"Epoch: {epoch + 1} | "
        f"train_loss: {train_loss:.4f} | "
        f"train_acc: {train_acc:.4f} | "
        f"test_loss: {test_loss:.4f} | "
        f"test_acc: {test_acc:.4f} | "
    )

    # Update results dictionary
    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)

  return results

## 5. Creating a function to save the model

In [ ]:
from pathlib import Path
def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name: str):

  # Create target directory
  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True, exist_ok=True)

  # Create model save path
  assert model_name.endswith(".pth") or model_name.endswith(".pt"), "model_name should end with `.pth` or `.pt`"
  model_save_path = target_dir_path / model_name

  # Save the model state_dict()
  print(f"[INFO] saving model to: {model_save_path}")
  torch.save(obj=model.state_dict(),
             f=model_save_path)

##6. Train, evaluate, and save the model

In [ ]:
# Set random seeds
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Set number of epochs
NUM_EPOCHS = 5

# Recreate an instance of TinyVGG
model_0 = TinyVGG(input_shape=3,
                  hidden_units=10,
                  output_shape=len(train_data.classes)).to(device)

# Setup loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_0.parameters(),
                            lr=0.001)

# Start the timer
from timeit import default_timer as timer
start_time = timer()

# Train model_0
model_0_results = train(model=model_0,
                        train_dataloader=train_dataloader,
                        test_dataloader=test_dataloader,
                        loss_fn=loss_fn,
                        optimizer=optimizer,
                        epochs=NUM_EPOCHS,
                        device=device)

# End the timer  and print how long it took
end_time = timer()
print(f"[INFO] Total training time: {end_time-start_time} second.")

# Save the model
save_model(model=model_0,
           target_dir="models",
           model_name="model_0.pth")

 This code is running one cell at once, now we will make it in a script mode.